In [3]:
import pandas as pd
print(pd.__version__)

2.3.3


In [5]:
df = pd.read_csv("diabetic_data.csv", na_values="?")
print(df.shape)
print(df["readmitted"].value_counts())
print(df.isna().mean().sort_values(ascending=False).head(10))
print(df["patient_nbr"].nunique(), "unique patients")

/var/folders/s4/xrl7yr8s5mxfv7j3gjsxw9980000gn/T/ipykernel_22992/1949893524.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("diabetic_data.csv", na_values="?")


(101766, 50)
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64
weight               0.968585
max_glu_serum        0.947468
A1Cresult            0.832773
medical_specialty    0.490822
payer_code           0.395574
race                 0.022336
diag_3               0.013983
diag_2               0.003518
diag_1               0.000206
encounter_id         0.000000
dtype: float64
71518 unique patients


In [6]:
df["target"] = (df["readmitted"] == "<30").astype(int)
print(df["target"].mean())   # roughly 0.11

# Patients who died or went to hospice can't be readmitted
bad_dispositions = [11, 13, 14, 19, 20, 21]
df = df[~df["discharge_disposition_id"].isin(bad_dispositions)]

0.11159915885462728


In [7]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, df["target"], groups=df["patient_nbr"]))
train, test = df.iloc[train_idx], df.iloc[test_idx]

assert set(train["patient_nbr"]).isdisjoint(test["patient_nbr"])